# SENSEI — Session Intelligence
## Module 1: Purchase Prediction · Exploratory Data Analysis

> SENSEI turns raw e-commerce clickstream data into session-level intelligence.  
> This notebook explores the raw RetailRocket event log to understand visitor behaviour  
> before building the SENSEI session feature store.

**Dataset:** [RetailRocket E-Commerce Dataset](https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset)

## Contents
1. Load & inspect data
2. Event distribution
3. Bot detection & removal
4. Visitor behaviour & conversion funnel
5. Time patterns
6. Key findings & feature motivation

## Key Findings

| # | Finding |
|---|---|
| 1 | 2,756,101 events from 1,407,580 unique visitors, May–Sep 2015 |
| 2 | Event funnel: ~97 % view → ~2.5 % add-to-cart → ~0.8 % transaction |
| 3 | ~1 % of visitors complete a purchase — strong class imbalance in the target |
| 4 | Biggest funnel drop: add-to-cart → purchase (~68 % of cart-adders do not buy) |
| 5 | Timestamps are UTC; RetailRocket is Russia-based, actual timezone likely UTC+3 |
| 6 | One `transactionid` can cover multiple items — one purchase ≠ one event row |

**Features motivated by this EDA:**

| Feature | Motivation |
|---|---|
| `n_views`, `n_addtocart` | Engagement depth |
| `n_items` | Breadth of interest |
| `n_revisited_items` | Re-engagement / purchase intent |
| `view_to_cart_ratio` | Visitor decisiveness |
| `duration_sec` | Session length |
| `hour_of_day`, `day_of_week` | Temporal purchase patterns |
| `is_first_session` | First-time vs. returning visitor |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.append(os.path.join('..', 'src'))

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Load & Inspect

In [ ]:
DATA_DIR = os.path.join('..', 'data')

df = pd.read_csv(os.path.join(DATA_DIR, 'events.csv'))
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')

print(f'Shape before deduplication: {df.shape}')
n_dupes = df.duplicated().sum()
print(f'Duplicate rows            : {n_dupes:,}')
df = df.drop_duplicates().reset_index(drop=True)
print(f'Shape after deduplication : {df.shape}')
df.head()

In [ ]:
# Bot / outlier detection
# Visitors with an extreme number of events are likely crawlers or test accounts.
events_per_visitor = df.groupby('visitorid').size()

threshold = 500
n_bots = (events_per_visitor > threshold).sum()
pct_bots = n_bots / len(events_per_visitor) * 100

print(f'Visitors with > {threshold} events : {n_bots:,} ({pct_bots:.2f} % of visitors)')
print(f'Events from these visitors         : {events_per_visitor[events_per_visitor > threshold].sum():,}')

fig, ax = plt.subplots(figsize=(10, 3))
sns.histplot(events_per_visitor.clip(upper=200), bins=100, ax=ax, color='#4C72B0')
ax.axvline(threshold, color='red', linestyle='--', label=f'Bot threshold ({threshold})')
ax.set_xlabel('Events per visitor (capped at 200)')
ax.set_ylabel('Visitors')
ax.set_title('Events per visitor — identifying potential bots')
ax.legend()
plt.tight_layout()
plt.show()

# Remove bot visitors from the dataframe
bot_visitors = events_per_visitor[events_per_visitor > threshold].index
df = df[~df['visitorid'].isin(bot_visitors)].reset_index(drop=True)
print(f'\nShape after bot removal: {df.shape}')

In [ ]:
df.info()

In [ ]:
print('=== Missing values ===')
print(df.isnull().sum())
print()
print('Note: transactionid is only set for transaction events — NaN for views/addtocart is expected.')

## 2. Event Distribution

event_counts = df['event'].value_counts().reset_index()
event_counts.columns = ['event', 'count']
event_counts['pct (%)'] = (event_counts['count'] / len(df) * 100).round(2)

print(event_counts.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=event_counts, x='event', y='count', hue='event', palette='muted', legend=False, ax=ax)
ax.set_title('Event counts')
ax.set_ylabel('Number of events')
ax.set_xlabel('')
for i, row in event_counts.iterrows():
    ax.text(i, row['count'] * 1.01,
            f"{row['count']:,}\n({row['pct (%)']:.1f}%)",
            ha='center', fontsize=9)
plt.tight_layout()
plt.show()

**Finding:** The funnel is extremely skewed — 96.7 % views, 2.5 % add-to-cart, 0.8 % transactions. This class imbalance will be the main challenge for modelling.

## 3. Visitor Behaviour

In [ ]:
events_per_visitor = df.groupby('visitorid').size()

print('Events per visitor:')
print(events_per_visitor.describe().round(1))

In [ ]:
# Visitors who purchased vs. never purchased
buyers = df[df['event'] == 'transaction']['visitorid'].unique()
print(f'Visitors who purchased at least once : {len(buyers):,}')
print(f'Visitors who never purchased          : {df["visitorid"].nunique() - len(buyers):,}')
print(f'Purchase conversion rate (visitors)   : {len(buyers) / df["visitorid"].nunique() * 100:.2f} %')

In [ ]:
all_visitors  = set(df['visitorid'].unique())
viewed        = set(df[df['event'] == 'view']['visitorid'].unique())
added_to_cart = set(df[df['event'] == 'addtocart']['visitorid'].unique())
transacted    = set(df[df['event'] == 'transaction']['visitorid'].unique())

funnel = pd.DataFrame({
    'Stage': ['Viewed', 'Added to cart', 'Purchased'],
    'Visitors': [len(viewed), len(added_to_cart), len(transacted)],
})
funnel['% of total'] = (funnel['Visitors'] / len(all_visitors) * 100).round(2)
funnel['% of prev stage'] = [
    100.0,
    len(added_to_cart) / len(viewed) * 100,
    len(transacted) / len(added_to_cart) * 100,
]
funnel['% of prev stage'] = funnel['% of prev stage'].round(2)
print(funnel.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=funnel, x='Stage', y='Visitors', hue='Stage', palette='muted', legend=False, ax=ax)
ax.set_title('Visitor conversion funnel')
ax.set_ylabel('Unique visitors')
ax.set_xlabel('')
for i, row in funnel.iterrows():
    ax.text(i, row['Visitors'] * 1.01,
            f"{row['Visitors']:,}\n({row['% of total']:.1f}%)",
            ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
df['hour']    = df['timestamp'].dt.hour
df['weekday'] = df['timestamp'].dt.day_name()

WEEKDAY_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

hourly = df.groupby('hour').size().reset_index(name='count')
daily  = (
    df.groupby('weekday').size()
    .reindex(WEEKDAY_ORDER)
    .reset_index(name='count')
)
daily['weekday_short'] = daily['weekday'].str[:3]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.lineplot(data=hourly, x='hour', y='count', marker='o', ax=axes[0], color='#4C72B0')
axes[0].set_title('Events by hour of day\n(UTC — likely UTC+3 in local time)')
axes[0].set_xlabel('Hour (UTC)')
axes[0].set_ylabel('Events')
axes[0].set_xticks(range(0, 24, 2))

sns.barplot(data=daily, x='weekday_short', y='count', hue='weekday_short',
            palette='muted', legend=False, ax=axes[1],
            order=[d[:3] for d in WEEKDAY_ORDER])
axes[1].set_title('Events by weekday')
axes[1].set_xlabel('')
axes[1].set_ylabel('Events')

plt.tight_layout()
plt.show()

In [ ]:
df['hour']    = df['timestamp'].dt.hour
df['weekday'] = df['timestamp'].dt.day_name()
df['week']    = df['timestamp'].dt.isocalendar().week.astype(int)

WEEKDAY_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

hourly = df.groupby('hour').size().reset_index(name='count')
daily  = (
    df.groupby('weekday').size()
    .reindex(WEEKDAY_ORDER)
    .reset_index(name='count')
)
daily['weekday_short'] = daily['weekday'].str[:3]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.lineplot(data=hourly, x='hour', y='count', marker='o', ax=axes[0], color='#4C72B0')
axes[0].set_title('Events by hour of day')
axes[0].set_xlabel('Hour (UTC)')
axes[0].set_ylabel('Events')
axes[0].set_xticks(range(0, 24, 2))

sns.barplot(data=daily, x='weekday_short', y='count', ax=axes[1], palette='muted',
            order=[d[:3] for d in WEEKDAY_ORDER])
axes[1].set_title('Events by weekday')
axes[1].set_xlabel('')
axes[1].set_ylabel('Events')

plt.tight_layout()
plt.show()